## 📑 Core Liquidations
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AQAUH)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, regexp_replace, array, col, explode, lit, when, split
import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim
import boto3
import os

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

ref_month_last_day = spark.sql(f'''SELECT last_day(add_months('{run_date}', -1)) AS SOM''').collect()[0]['SOM']

ref_date = spark.sql(f'''
                         select max(NextBusinessDay) MaxBD1
                         from `{env_prefix}-cdh`.reference.ref_calendar
                         where MONTH(calendardate)=MONTH(add_months('{run_date}', -1)) AND YEAR(calendardate) =YEAR(add_months('{run_date}', -1))
                         AND MONTH(NextBusinessDay)=MONTH(add_months('{run_date}', -1)) AND YEAR(NextBusinessDay) =YEAR(add_months('{run_date}', -1))
                         ''').collect()[0]['MaxBD1']
ref_date = ref_date.strftime('%Y-%m-%d') 

som_month_last_day = spark.sql(f'''SELECT last_day(add_months('{run_date}', -2)) AS SOM''').collect()[0]['SOM']
  
som_date = spark.sql(f'''
                         select max(NextBusinessDay) MaxBD2
                         from `{env_prefix}-cdh`.reference.ref_calendar
                         where MONTH(calendardate)=MONTH(add_months('{run_date}', -2)) AND YEAR(calendardate) =YEAR(add_months('{run_date}', -2))
                         AND MONTH(NextBusinessDay)=MONTH(add_months('{run_date}', -2)) AND YEAR(NextBusinessDay) =YEAR(add_months('{run_date}', -2))
                         ''').collect()[0]['MaxBD2']
som_date = som_date.strftime('%Y-%m-%d') 

EXPRT_MONTH_EOM = spark.sql(f'''
                    SELECT date_format(last_day(add_months('{run_date}', -1)),'yyyyMM') AS Months
                         ''').collect()[0]['Months']
EXPRT_MONTH_SOM = spark.sql(f'''
                    SELECT date_format(last_day(add_months('{run_date}', -2)),'yyyyMM') AS Months
                         ''').collect()[0]['Months']

###### Schema of the table

In [0]:
table_name = 'Core_Liquidations'

schema = (
"BusinessDate date," 
"ReferenceMonth string,"
"AssetSegment string,"
"PartyTaxRegistrationNumber string,"
"ExposureKey string,"
"AgreementKey string,"
"SPVDesc string,"   
"PaymentAmount string,"
"PaymentDate string,"
"AccountStatusPrv string,"
"PaymentReasonDescription string,"
"AuctionDate string,"
"UniqueAuctionCode string,"
"SuccessfulTenderer string,"
"Hastener string,"
"NotFoundInPayments string,"
"ProgrammeNumber string,"
"ExecutionCostsAttributedToHastenerSPVAlphaBank string,"
"TotalRankAmount decimal(19,2),"
"FinalTotalRankAmount decimal(19,2),"
"InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    ref_date,  # type: ignore
)


##### Create Delta Tables with selected fields & Temporary Views for each delta table

##### Liquidations

In [0]:
Liquidations = spark.sql(
f'''
    SELECT	
				ExposureKey
			,	PartyTaxRegistrationNumber
			,	ProgramNumber
			, date_format(to_date(LEFT(AuctionDate,10), 'd/M/yyyy'),'yyyy-MM-dd')  AS AuctionDate
			, date_format(to_date(LEFT(CreditDate,10), 'd/M/yyyy'),'yyyy-MM-dd')  AS CreditDate
			,	Hastener
			,	UniqueId
			,	FirstBidder
			,	ConfirmedCreditAmount
    FROM `{env_prefix}-cdh`.externalfiles.extauctionrecoveries
    where BusinessDate = {ref_date!r}
    and IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extauctionrecoveries WHERE BusinessDate = {ref_date!r})
'''
)
Liquidations.createOrReplaceTempView("Liquidations")

# Check if the dataset is empty and print a message accordingly
if Liquidations.count() == 0:
    print(f"The required external file extauctionrecoveries dataset for date {ref_date!r} is empty")


##### mastertable

In [0]:
mastertable = spark.sql(
f'''
    SELECT ExposureKey
        , AgreementKey
        , Bucket
        , AccountStatus as ArrearsStatus
        , A.EBAStatus 
        , a.SPV
    FROM `{env_prefix}-cdh-delivery`.repoint.core_mastertable a
    where BusinessDate = {som_date!r}
'''
)
mastertable.createOrReplaceTempView("mastertable")


##### qualcoSoM

In [0]:
qualcoSoM = spark.sql(
f'''
    SELECT ACCOUNT_NUMBER
         , Account_Status_Final
         , CURRENT_BUCKET
         , SPV_desc
         , CONTRACT_NO
         , BU
         , EXPRT_MONTH
    FROM `{env_prefix}-cdh-delivery`.repoint.core_retailprimary 
    WHERE EXPRT_MONTH = '{EXPRT_MONTH_SOM}' 
    AND BusinessDate = (SELECT MAX(BusinessDate) BusinessDate FROM `{env_prefix}-cdh-delivery`.repoint.core_retailprimary WHERE EXPRT_MONTH = '{EXPRT_MONTH_SOM}') 
'''
)   
qualcoSoM.createOrReplaceTempView("qualcoSoM")

##### qualcoEoM

In [0]:
qualcoEoM = spark.sql(
f'''
    SELECT ACCOUNT_NUMBER
         , Account_Status_Final
         , CURRENT_BUCKET
         , SPV_desc
         , CONTRACT_NO
         , BU
         , EXPRT_MONTH
    FROM `{env_prefix}-cdh-delivery`.repoint.core_retailprimary 
    WHERE EXPRT_MONTH = '{EXPRT_MONTH_EOM}' 
    AND BusinessDate = (SELECT MAX(BusinessDate) BusinessDate FROM `{env_prefix}-cdh-delivery`.repoint.core_retailprimary WHERE EXPRT_MONTH = '{EXPRT_MONTH_EOM}') 
'''
)   
qualcoEoM.createOrReplaceTempView("qualcoEoM")

##### nfpayments

In [0]:
nfpayments = spark.sql(
f'''
SELECT * 
FROM `{env_prefix}-cdh-delivery`.repoint.nfpayments
WHERE businessdate = (SELECT MAX(BusinessDate) from `{env_prefix}-cdh-delivery`.repoint.nfpayments 
                      WHERE last_day(BusinessDate) = last_day({run_date!r}))
'''
)
nfpayments.createOrReplaceTempView("nfpayments")


##### Payments

In [0]:
Payments = spark.sql(
f'''
		SELECT	p.BUSINNES_UNIT
			,	p.ACCOUNT_NUMBER
			,	p.CONTRACT_NUMBER
			,	p.PAY_DATE
			,	p.PAY_AMT_LCY
			,	p.PAYMENT_REASON_DESCRIPTION
			,	p.PAY_CODE	
			,	SUM(PAY_AMT_LCY) OVER (PARTITION BY p.ACCOUNT_NUMBER, P.PAY_DATE) AS PAY_AMT_LCY_final
		FROM nfpayments AS P 
		WHERE 1=1
'''
)
Payments.createOrReplaceTempView("Payments")



##### Payments_v2

In [0]:
Payments_v2 = spark.sql(
f'''
	SELECT	p.*
	,	RANK() OVER (PARTITION BY p.ACCOUNT_NUMBER, P.PAY_DATE, p.PAY_AMT_LCY_final ORDER BY P.PAYMENT_REASON_DESCRIPTION ASC, PAY_CODE ASC)  AS RnkAccDateTotalAmount
	,	RANK() OVER (PARTITION BY p.ACCOUNT_NUMBER, P.PAY_DATE, p.PAY_AMT_LCY ORDER BY P.PAYMENT_REASON_DESCRIPTION ASC, PAY_CODE ASC)		  AS RnkAccDateAmount
	,	RANK() OVER (PARTITION BY p.ACCOUNT_NUMBER, p.PAY_AMT_LCY ORDER BY P.PAYMENT_REASON_DESCRIPTION ASC, PAY_CODE ASC)	   				  AS RnkAccAmount
	FROM Payments P
'''
)
Payments_v2.createOrReplaceTempView("Payments_v2")



##### Liquidations

In [0]:
Liquidations = spark.sql(
f'''
		SELECT	
				L.ExposureKey
			,	L.PartyTaxRegistrationNumber
			,	L.ProgramNumber
			,	L.AuctionDate
			,	L.CreditDate 																AS PaymentDate
			,	L.Hastener
			,	L.UniqueId
			,	L.FirstBidder
			,	SUM(TRY_CAST(replace(L.ConfirmedCreditAmount, ',', '.') AS DECIMAL(19,2))) 	AS PaymentAmount
		FROM Liquidations L		
		WHERE 1=1 
			  AND L.ExposureKey IS NOT NULL
		GROUP BY 
				L.ExposureKey
			,	L.PartyTaxRegistrationNumber
			,	L.ProgramNumber
			,	L.AuctionDate
			,	L.CreditDate 							
			,	L.Hastener
			,	L.UniqueId
			,	L.FirstBidder
'''
)
Liquidations.createOrReplaceTempView("Liquidations")


##### Eisprakseis

In [0]:
Eisprakseis = spark.sql(
f'''
		SELECT 
				E.UniqueId
			,	SUM(CASE WHEN e.ExecutionExpense = 'NULL' THEN 0 ELSE CAST(e.ExecutionExpense AS DECIMAL(19,2)) END) AS ExecutionExpense
			,	SUM(CASE WHEN E.TotalAmount = 'NULL' THEN 0 ELSE CAST(E.TotalAmount AS DECIMAL(19,2)) END) AS TotalAmount
		FROM `{env_prefix}-cdh`.externalfiles.extrecoveries_total	AS E	
    WHERE BusinessDate = {ref_date!r}
    AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extrecoveries_total WHERE BusinessDate = {ref_date!r})
		GROUP BY E.UniqueId
''')
Eisprakseis.createOrReplaceTempView("Eisprakseis")

# Check if the dataset is empty and print a message accordingly
if Eisprakseis.count() == 0:
    print(f"The required external file extrecoveries_total for dates {ref_date!r} is empty")


##### Liquidations_V2

In [0]:
Liquidations_V2 = spark.sql(
f'''
		SELECT	
				L.*
			,	CASE WHEN P1.ACCOUNT_NUMBER IS NOT NULL	THEN P1.PAYMENT_REASON_DESCRIPTION
					 WHEN P2.ACCOUNT_NUMBER IS NOT NULL		THEN P2.PAYMENT_REASON_DESCRIPTION
					 WHEN P3.ACCOUNT_NUMBER IS NOT NULL		THEN P3.PAYMENT_REASON_DESCRIPTION
				END AS PaymentReasonDescription
			,   e.ExecutionExpense
			,	e.TotalAmount
			,	e.TotalAmount - CASE WHEN e.ExecutionExpense IS NULL THEN '0'
                                              ELSE e.ExecutionExpense END AS FinalTotalAmount
			,	CASE WHEN P1.ACCOUNT_NUMBER IS NOT NULL	THEN P1.CONTRACT_NUMBER
					 WHEN P2.ACCOUNT_NUMBER IS NOT NULL		THEN P2.CONTRACT_NUMBER
					 WHEN P3.ACCOUNT_NUMBER IS NOT NULL		THEN P3.CONTRACT_NUMBER
					 WHEN PSoM.ACCOUNT_NUMBER IS NOT NULL	THEN PSoM.CONTRACT_NO
				END AS ContractNumber
			,	CASE WHEN P1.ACCOUNT_NUMBER IS NOT NULL	THEN RIGHT(P1.BUSINNES_UNIT,2)
					 WHEN P2.ACCOUNT_NUMBER IS NOT NULL		THEN RIGHT(P2.BUSINNES_UNIT,2)
					 WHEN P3.ACCOUNT_NUMBER IS NOT NULL		THEN RIGHT(P3.BUSINNES_UNIT,2)
					 WHEN PSoM.ACCOUNT_NUMBER IS NOT NULL	THEN RIGHT(PSoM.BU,2)
				END AS BusinessUnit
			,	CASE WHEN NTP.ExposureKey IS NOT NULL			THEN NTP.ArrearsStatus
					 WHEN PSoM.ACCOUNT_NUMBER IS NOT NULL			THEN PSoM.Account_Status_Final
					 WHEN PeoM.ACCOUNT_NUMBER IS NOT NULL			THEN PEoM.Account_Status_Final
				end as AccountStatus
			,	CASE WHEN NTP.ExposureKey IS NOT NULL			THEN NTP.SPV
					 WHEN PSoM.ACCOUNT_NUMBER IS NOT NULL			THEN PSoM.SPV_DESC
					 WHEN PeoM.ACCOUNT_NUMBER IS NOT NULL			THEN PeoM.SPV_DESC
				end as SPVDesc
			,	CASE WHEN P1.ACCOUNT_NUMBER IS NULL	AND P2.ACCOUNT_NUMBER IS NULL AND P3.ACCOUNT_NUMBER IS NULL	THEN 'Yes'
					 END AS NotFoundInPayments
		FROM Liquidations	 AS L
		LEFT JOIN Payments_V2 AS P1	ON P1.ACCOUNT_NUMBER = L.ExposureKey AND P1.PAY_DATE = L.PaymentDate AND P1.PAY_AMT_LCY_final = L.PaymentAmount	AND P1.RnkAccDateTotalAmount = 1
		LEFT JOIN Payments_V2 AS P2	ON P2.ACCOUNT_NUMBER = L.ExposureKey AND P2.PAY_DATE = L.PaymentDate AND P2.PAY_AMT_LCY = L.PaymentAmount	AND P2.RnkAccDateAmount = 1
		LEFT JOIN Payments_V2 AS P3 ON P3.ACCOUNT_NUMBER = L.ExposureKey AND P3.PAY_AMT_LCY = L.PaymentAmount	AND P3.RnkAccAmount = 1
		LEFT JOIN Eisprakseis AS E		ON E.UniqueId= L.UniqueId
		LEFT JOIN mastertable AS NTP	ON NTP.ExposureKey = L.ExposureKey
		LEFT JOIN qualcoSoM	  AS PSoM	ON PSoM.ACCOUNT_NUMBER = L.ExposureKey
		LEFT JOIN qualcoEoM	  AS PEoM	ON PEoM.ACCOUNT_NUMBER = L.ExposureKey
'''
)
Liquidations_V2.createOrReplaceTempView("Liquidations_V2")


In [0]:
final_df = spark.sql(
f'''
		SELECT DISTINCT
				CAST('{ref_date}' AS DATE)  																	AS BusinessDate
			,	CAST(CONCAT(CAST(YEAR('{ref_date}') AS VARCHAR(4)), 
				CASE WHEN len(CAST(MONTH('{ref_date}') AS VARCHAR(2)))=1 
     				THEN CONCAT('0',CAST(MONTH('{ref_date}') AS VARCHAR(2))) 
      				ELSE CAST(MONTH('{ref_date}') AS VARCHAR(2)) 
     			 END) AS VARCHAR(6))											 								AS ReferenceMonth
			,	CAST(L.BusinessUnit AS STRING) 																	AS AssetSegment
			,	CAST(L.PartyTaxRegistrationNumber AS STRING) 													AS PartyTaxRegistrationNumber
			,	CAST(ExposureKey AS STRING) 																	AS ExposureKey
			, 	CAST(ContractNumber AS STRING) 																	AS AgreementKey
			,	CAST(SPVDesc AS STRING) 																		AS SPVDesc
			,	CAST(PaymentAmount AS STRING) 																	AS PaymentAmount
			,	CAST(PaymentDate AS STRING)																		AS PaymentDate
			,	CAST(AccountStatus AS STRING) 																	AS AccountStatusPrv
			,	CAST(PaymentReasonDescription AS STRING) 														AS PaymentReasonDescription
			,	CAST(AuctionDate AS STRING) 																	AS AuctionDate
			,	CAST(UniqueId AS STRING) 																		AS UniqueAuctionCode
			,	CAST(FirstBidder AS STRING) 	 																AS SuccessfulTenderer
			,	CAST(Hastener AS STRING)  																		AS Hastener
			,	CAST(NotFoundInPayments AS STRING) 																AS NotFoundInPayments
			,	CAST(ProgramNumber AS STRING)																	AS ProgrammeNumber
			,	CAST(ExecutionExpense AS STRING)																AS ExecutionCostsAttributedToHastenerSPVAlphaBank
			,	CAST(TotalAmount AS DECIMAL(19,2))  															AS TotalRankAmount
			,	CAST(FinalTotalAmount AS DECIMAL(19,2)) 														AS FinalTotalRankAmount
		  , CAST(getdate() AS TIMESTAMP) 																		AS InsertTimeStamp
		FROM Liquidations_V2 AS L
  '''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_Liquidations: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = []
check_table(delivery_table_path, ref_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
Liquidations = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {ref_date!r}
'''
)
Liquidations.createOrReplaceTempView("Liquidations")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/Liquidations/"
file_name = "Liquidations_" +ref_date+ ".csv"
Liquidations.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)